In [1]:
pip install torch transformers pandas scikit-learn matplotlib seaborn tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
# 修正 1: 從 torch.optim 匯入 AdamW，而不是 transformers
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import time
import matplotlib.font_manager as fm
from tqdm import tqdm # 進度條

# 設定 Mac 專用中文字型
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC', 'sans-serif'] 
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 1. 設定區 (Config)
# ==========================================
TRAIN_PKL = 'train_seg.pkl'
TEST_PKL = 'test_seg.pkl'
MODEL_NAME = 'hfl/chinese-roberta-wwm-ext' # 哈工大中文 RoBERTa
MAX_LEN = 256       # 文本最大長度
BATCH_SIZE = 32     # M4 Pro 性能強，設 32 加快速度 (若報錯請改回 16)
EPOCHS = 3          # 訓練輪數
LEARNING_RATE = 2e-5 

# 🚀 M4 Pro 核心設定：啟用 Metal Performance Shaders (MPS)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 檢測到 Apple Silicon (M4 Pro)！已啟用 MPS GPU 全速加速模組。")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 檢測到 NVIDIA GPU，已啟用 CUDA 加速。")
else:
    device = torch.device("cpu")
    print("⚠️ 未檢測到 GPU，將使用 CPU (速度較慢)。")

# ==========================================
# 2. 資料準備 (Dataset)
# ==========================================
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_data_loader(df, tokenizer, max_len, batch_size):
    ds = NewsDataset(
        texts=df['text_clean'].to_numpy(),
        labels=df['label_idx'].to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    # Mac 上 num_workers 必須設為 0 以避免多進程錯誤
    return DataLoader(ds, batch_size=batch_size, num_workers=0)

# ==========================================
# 3. 評估函式 (Evaluation)
# ==========================================
def evaluate_model(model, data_loader, device, label_names, training_time):
    model = model.eval()
    
    predictions = []
    prediction_probs = []
    real_values = []

    # 顯示評估進度條
    print("正在評估測試集...")
    with torch.no_grad():
        for d in tqdm(data_loader, desc="Testing"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1) # 轉成機率
            preds = torch.argmax(probs, dim=1)

            predictions.extend(preds.cpu().tolist())
            prediction_probs.extend(probs.cpu().tolist())
            real_values.extend(labels.cpu().tolist())

    # 計算指標
    y_true = np.array(real_values)
    y_pred = np.array(predictions)
    y_proba = np.array(prediction_probs)

    # 1. 整體 Weighted 指標
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    
    try:
        auroc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
    except:
        auroc = 0.0

    # 2. 詳細報表
    class_report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    class_report_df = pd.DataFrame(class_report).transpose()

    # 3. 輸出結果
    print("\n" + "="*80)
    print("🏆 RoBERTa 模型評估結果 (Model Ranking Summary)")
    print("="*80)
    
    summary_df = pd.DataFrame([{
        'Model': 'RoBERTa-wwm-ext (M4 Pro)',
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'AUROC': auroc,
        'Time(s)': training_time
    }])
    print(summary_df.to_string(float_format="{:.4f}".format, index=False))
    print("\n")

    print("="*80)
    print(f"📊 模型詳情: RoBERTa-wwm-ext")
    print("="*80)
    print(f"訓練時間: {training_time:.2f} 秒")
    print(f"整體 AUROC: {auroc:.4f}")
    print("\n[ 詳細分類指標表 ]")
    print(class_report_df.to_string(float_format="{:.4f}".format))

    # 4. 混淆矩陣圖
    print(f"\n[ 混淆矩陣圖 - RoBERTa ]")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
    plt.title(f'RoBERTa Confusion Matrix\nAcc: {acc:.4f} | F1: {f1:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

# ==========================================
# 主程式
# ==========================================
if __name__ == '__main__':
    # 1. 資料讀取與處理
    print("1. 讀取資料並還原文本 (Remove Monpa Spaces)...")
    try:
        train_df = pd.read_pickle(TRAIN_PKL)
        test_df = pd.read_pickle(TEST_PKL)
    except FileNotFoundError:
        print("❌ 找不到 train_seg.pkl 或 test_seg.pkl，請先執行 01_preprocessing.py")
        exit()

    # 填補空值
    train_df['text'] = train_df['text'].fillna('')
    test_df['text'] = test_df['text'].fillna('')

    # 把 Monpa 的空白拿掉，讓 BERT 讀原始句子
    train_df['text_clean'] = train_df['text'].apply(lambda x: x.replace(' ', ''))
    test_df['text_clean'] = test_df['text'].apply(lambda x: x.replace(' ', ''))

    # Label Encoding
    le = LabelEncoder()
    train_df['label_idx'] = le.fit_transform(train_df['label'])
    test_df['label_idx'] = le.transform(test_df['label'])
    label_names = le.classes_
    num_classes = len(label_names)
    print(f"   類別數量: {num_classes} {label_names}")

    # 2. Tokenizer 與 DataLoader
    print(f"2. 載入 Tokenizer: {MODEL_NAME}")
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    
    train_loader = create_data_loader(train_df, tokenizer, MAX_LEN, BATCH_SIZE)
    test_loader = create_data_loader(test_df, tokenizer, MAX_LEN, BATCH_SIZE)

    # 3. 模型初始化
    print("3. 下載並初始化 RoBERTa 模型...")
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes
    )
    model = model.to(device)

    # 修正 2: 優化器與排程器 (改用 torch.optim.AdamW)
    # 注意：torch.optim.AdamW 沒有 correct_bias 參數，所以這裡拿掉它
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE) 
    
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # 4. 開始訓練
    print(f"\n4. M4 Pro 全速訓練開始 (Epochs: {EPOCHS})...")
    start_time = time.time()
    
    for epoch in range(EPOCHS):
        print(f'\nEpoch {epoch + 1}/{EPOCHS}')
        print('-' * 10)

        model.train()
        losses = []
        correct_predictions = 0

        # 使用 tqdm 顯示進度條
        loop = tqdm(train_loader, leave=True)
        for d in loop:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits
            
            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            loop.set_description(f"Training Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())

        train_acc = correct_predictions.float() / len(train_df) 
        print(f"   Train Loss: {np.mean(losses):.4f} | Train Acc: {train_acc:.4f}")

    total_training_time = time.time() - start_time
    print(f"\n訓練完成！總耗時: {total_training_time:.2f} 秒")

    # 5. 評估與輸出報表
    print("\n5. 執行測試集評估...")
    evaluate_model(model, test_loader, device, label_names, total_training_time)

🚀 檢測到 Apple Silicon (M4 Pro)！已啟用 MPS GPU 全速加速模組。
1. 讀取資料並還原文本 (Remove Monpa Spaces)...
   類別數量: 8 ['feel_angry' 'feel_boring' 'feel_depressing' 'feel_happy'
 'feel_informative' 'feel_odd' 'feel_warm' 'feel_worried']
2. 載入 Tokenizer: hfl/chinese-roberta-wwm-ext
3. 下載並初始化 RoBERTa 模型...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/chinese-roberta-wwm-ext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



4. M4 Pro 全速訓練開始 (Epochs: 3)...

Epoch 1/3
----------


Training Epoch 1:   6%|▊           | 75/1182 [01:38<23:41,  1.28s/it, loss=1.09]